# Etapa 1 — Learning to Rank

## Consolidación y creación de labels

### Muestra de 500 usuarios por recursos limitados de PC
### Label 1, 2 y 0

# Justificar cuántos negativos por positivo se usaron y por qué.

### 2 negativos por cada 1 positivo, es decir un Ratio 2:1

* En el e-commerce real, la relación entre lo que un usuario ve (o podría comprar) y lo que realmente compra es de miles a uno. Si metemos todos los productos del catálogo como negativos, el modelo se volvería "perezoso" y predeciría siempre 0.
* El archivo order_products__prior.csv tiene más de 32 millones de filas. Si por cada producto comprado (positivo) agregamos 10 o 20 negativos, el dataset final superará los 100 millones de filas, haciendo imposible entrenar el modelo en una computadora local.
* Al elegir negativos del mismo departamento que el usuario ya consume, obligamos al ranker a volverse "fino".

In [3]:
import os
import pandas as pd
import numpy as np
import gc

print("Iniciando Pipeline con consumo de RAM Ultra-Bajo")

ruta_carpeta = r"D:\Tipti\Data"
NUM_USUARIOS_MUESTRA = 500 # Recursos limitados 
RATIO_NEGATIVOS = 2
np.random.seed(42)

# Carga de tablas
df_products = pd.read_csv(os.path.join(ruta_carpeta, "products.csv"))
df_orders = pd.read_csv(os.path.join(ruta_carpeta, "orders.csv"))

# Seleccionamos los 500 usuarios muestra DIRECTO desde la tabla de órdenes
todos_los_usuarios = df_orders['user_id'].unique()
usuarios_muestra = np.random.choice(todos_los_usuarios, size=NUM_USUARIOS_MUESTRA, replace=False)

# Filtramos df_orders para que SOLO tenga los datos de nuestros 500 usuarios
df_orders_filtrado = df_orders[df_orders['user_id'].isin(usuarios_muestra)].copy()
ordenes_de_la_muestra = set(df_orders_filtrado['order_id'].unique())

#print(f" Muestra {NUM_USUARIOS_MUESTRA} usuarios con {len(ordenes_de_la_muestra)} órdenes en total")

#Leamos el archivo más grande por pedazos para optimizar la RAM
print("Procesando 'order_products__prior.csv' en porciones para proteger la RAM")

chunk_size = 1_000_000  # Lee de 1 en 1 millón de filas 
lista_chunks_positivos = []

ruta_prior = os.path.join(ruta_carpeta, "order_products__prior.csv")

for chunk in pd.read_csv(ruta_prior, chunksize=chunk_size):
    chunk_filtrado = chunk[chunk['order_id'].isin(ordenes_de_la_muestra)]
    if not chunk_filtrado.empty:
        lista_chunks_positivos.append(chunk_filtrado)

# Unimos 
df_prior_filtrado = pd.concat(lista_chunks_positivos, ignore_index=True)

#print(f"Dataset reducido a: {len(df_prior_filtrado)} filas.")

# Forzar limpieza inicial de memoria
del lista_chunks_positivos
gc.collect()

#___________________________________________________________________________________________
#LABEL 1 Y 2
df_user_products = pd.merge(
    df_prior_filtrado[['order_id', 'product_id', 'reordered']], 
    df_orders_filtrado[['order_id', 'user_id']], 
    on='order_id', 
    how='inner'
)

df_positivos = df_user_products.groupby(['user_id', 'product_id']).agg(
    total_compras=('reordered', 'count'),
    fue_reordenado=('reordered', 'max')
).reset_index()

df_positivos['label'] = np.where(df_positivos['fue_reordenado'] == 1, 2, 1)

df_positivos = pd.merge(df_positivos, df_products[['product_id', 'department_id']], on='product_id', how='inner')

del df_prior_filtrado, df_user_products
gc.collect()

#_____________________________________________________________________________________
#LABEL 0
#print("Negative Sampling")

df_conteos = df_positivos.groupby(['user_id', 'department_id']).size().reset_index(name='num_positivos')
df_conteos['num_negativos_necesarios'] = df_conteos['num_positivos'] * RATIO_NEGATIVOS

# Cruzamos solo con la estructura esencial del catálogo
df_candidatos = pd.merge(df_conteos, df_products[['department_id', 'product_id']], on='department_id', how='inner')

# Remover lo que ya compró
df_candidatos = pd.merge(df_candidatos, df_positivos[['user_id', 'product_id']], on=['user_id', 'product_id'], how='left', indicator=True)
df_negativos_candidatos = df_candidatos[df_candidatos['_merge'] == 'left_only'].drop(columns=['_merge'])

del df_candidatos
gc.collect()

# Muestreo nativo por desorden
df_negativos_shuffled = df_negativos_candidatos.sample(frac=1, random_state=42).reset_index(drop=True)
del df_negativos_candidatos

df_negativos_shuffled['cum_count'] = df_negativos_shuffled.groupby(['user_id', 'department_id']).cumcount()
df_negativos = df_negativos_shuffled[df_negativos_shuffled['cum_count'] < df_negativos_shuffled['num_negativos_necesarios']].copy()

del df_negativos_shuffled
gc.collect()

# Estructurar Negativos
df_negativos = df_negativos[['user_id', 'product_id', 'department_id']]
df_negativos['total_compras'] = 0
df_negativos['fue_reordenado'] = 0
df_negativos['label'] = 0


# CONSOLIDACIÓN FINAL
df_dataset_final = pd.concat([df_positivos, df_negativos], ignore_index=True)

#print("\nFIN")
print(df_dataset_final['label'].value_counts().to_frame(name='Cantidad por Label'))

Iniciando Pipeline con consumo de RAM Ultra-Bajo
Procesando 'order_products__prior.csv' en porciones para proteger la RAM
       Cantidad por Label
label                    
0                   59718
1                   18113
2                   11746


In [4]:
print(df_dataset_final)

       user_id  product_id  total_compras  fue_reordenado  label  \
0          273         264              1               0      1   
1          273        1405              1               0      1   
2          273        2519              3               1      2   
3          273        2698              1               0      1   
4          273        3990              1               0      1   
...        ...         ...            ...             ...    ...   
89572   164825       18119              0               0      0   
89573    49553       42091              0               0      0   
89574   185845       28067              0               0      0   
89575   185845       45682              0               0      0   
89576   194850         503              0               0      0   

       department_id  
0                 16  
1                 19  
2                 14  
3                  7  
4                 16  
...              ...  
89572              4  

## Features

Como la columna department_id ya viene dada como números enteros (del 1 al 21), técnicamente ya cumple con ser un 'Label Encoding' nativo listo para modelos numéricos.
Solo nos aseguramos de que esté presente en todas las filas (incluyendo los negativos).

In [5]:
import pandas as pd
import numpy as np
import gc


df_user_features = df_positivos[['user_id', 'product_id', 'total_compras', 'fue_reordenado']].copy()
df_user_features = df_user_features.rename(columns={
    'total_compras': 'user_product_frequency',
    'fue_reordenado': 'user_product_reordered'
})

# Unimos a nuestro dataset unificado 
df_dataset_final = pd.merge(
    df_dataset_final, 
    df_user_features, 
    on=['user_id', 'product_id'], 
    how='left'
)

# (label=0) no tienen historial de compra
# Por lo tanto, tras el merge quedarán como NaN. Los rellenamos con 0.
df_dataset_final['user_product_frequency'] = df_dataset_final['user_product_frequency'].fillna(0).astype(int)
df_dataset_final['user_product_reordered'] = df_dataset_final['user_product_reordered'].fillna(0).astype(int)

del df_user_features
gc.collect()


# como estamos usando una muestra de 500 usuarios por limitaciones de PC vamos a calcular la poluparidad global de la siguiente forma:
#print("Calculando product_global_popularity...")
# Contamos cuántos usuarios únicos compraron cada producto en nuestro espacio de trabajo
df_popularity = df_positivos.groupby('product_id')['user_id'].nunique().reset_index(name='product_global_popularity')

# Unimos la popularidad global al dataset principal
df_dataset_final = pd.merge(df_dataset_final, df_popularity, on='product_id', how='left')
df_dataset_final['product_global_popularity'] = df_dataset_final['product_global_popularity'].fillna(0).astype(int)

del df_popularity
gc.collect()


# no queden valores vacíos
df_dataset_final['department_id'] = df_dataset_final['department_id'].fillna(0).astype(int)

columnas_finales = ['user_id', 'product_id', 'user_product_frequency', 'user_product_reordered', 'product_global_popularity', 'department_id', 'label']
print(df_dataset_final[columnas_finales].head(10))

   user_id  product_id  user_product_frequency  user_product_reordered  \
0      273         264                       1                       0   
1      273        1405                       1                       0   
2      273        2519                       3                       1   
3      273        2698                       1                       0   
4      273        3990                       1                       0   
5      273        3996                       1                       0   
6      273        6187                       3                       1   
7      273        6195                       2                       1   
8      273        7065                       1                       0   
9      273        7751                       1                       0   

   product_global_popularity  department_id  label  
0                          4             16      1  
1                          1             19      1  
2                         

# Modelo

In [6]:
import pandas as pd
import numpy as np
from xgboost import XGBRanker
import gc

print("Preparando los datos para el XGBRanker")


# todas las filas de un mismo usuario estén juntas
df_dataset_final = df_dataset_final.sort_values(by='user_id').reset_index(drop=True)

features = [
    'user_product_frequency', 
    'user_product_reordered', 
    'product_global_popularity', 
    'department_id'
]

X_train = df_dataset_final[features]
y_train = df_dataset_final['label']  


# VECTOR GROUP
# Contamos cuántas filas (productos candidatos) tiene cada usuario de forma ordenada
fit_groups = df_dataset_final.groupby('user_id').size().to_numpy()

print(f"Dataset listo. Total filas: {len(X_train)} | Total usuarios únicos (grupos): {len(fit_groups)}")

# Limpieza de RAM
gc.collect()


#  CONFIGURAR E INSTANCIAR EL MODELO
ranker = XGBRanker(
    objective='rank:ndcg',
    eval_metric='ndcg@10',  
    n_estimators=100,       # Número de árboles
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)


# ENTRENAR EL MODELO

print("\nEntrenando el XGBRanker")
ranker.fit(
    X_train, 
    y_train, 
    group=fit_groups,  
    verbose=True
)


Preparando los datos para el XGBRanker
Dataset listo. Total filas: 89577 | Total usuarios únicos (grupos): 500

Entrenando el XGBRanker


,objective,'rank:ndcg'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'ndcg@10'


In [1]:
%pip install pandas numpy xgboost pyarrow


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Evaluemos el Modelo

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score
import gc


ruta_train = os.path.join(ruta_carpeta, "order_products__train.csv")
lista_chunks_test = []

print("Cargando set de test filtrado por la muestra de usuarios...")
for chunk in pd.read_csv(ruta_train, chunksize=1_000_000):
    chunk_filtrado = chunk[chunk['order_id'].isin(ordenes_de_la_muestra)] # Reutiliza las órdenes de tu muestra
    if not chunk_filtrado.empty:
        lista_chunks_test.append(chunk_filtrado)

df_test_raw = pd.concat(lista_chunks_test, ignore_index=True)

# Mapeamos para obtener el user_id correspondiente a cada orden de test
df_test_real = pd.merge(df_test_raw[['order_id', 'product_id']], df_orders_filtrado[['order_id', 'user_id']], on='order_id', how='inner')

# Creamos un conjunto (set) de los productos reales comprados por usuario en test para una búsqueda rápida
ground_truth = df_test_real.groupby('user_id')['product_id'].apply(set).to_dict()

print(f"Ground truth estructurado para {len(ground_truth)} usuarios en el set de test.")
del df_test_raw, df_test_real, lista_chunks_test
gc.collect()


#  GENERAR SCORES DEL MODELO Y DEL BASELINE

# Usamos el mismo dataset final (X_train) para obtener las predicciones de ranking sobre los candidatos conocidos
df_eval = df_dataset_final[['user_id', 'product_id', 'product_global_popularity']].copy()
df_eval['model_score'] = ranker.predict(df_dataset_final[features])

# CALCULAR NDCG@10 Y MAP@10 

def calcular_map_at_k(relevancias_binarias, k=10):
    """Calcula el Average Precision en K (AP@K) para un usuario"""
    relevancias = relevancias_binarias[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(relevancias):
        if p > 0:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    if num_hits == 0:
        return 0.0
    return score / min(len(relevancias), num_hits)

ndcg_modelo, ndcg_baseline = [], []
map_modelo, map_baseline = [], []

# Evaluamos usuario por usuario dentro de nuestra muestra evaluable
usuarios_evaluables = [u for u in df_eval['user_id'].unique() if u in ground_truth]

for user in usuarios_evaluables:
    # Obtener candidatos del usuario
    df_user = df_eval[df_eval['user_id'] == user].copy()
    productos_reales = ground_truth[user]
    
    # Crear vector de relevancia real (1 si lo compró en test, 0 si no)
    df_user['relevance_real'] = df_user['product_id'].apply(lambda x: 1 if x in productos_reales else 0)
    
    # EVALUACIÓN MODELO 
    df_modelo_sorted = df_user.sort_values(by='model_score', ascending=False).head(10)
    y_true_modelo = df_modelo_sorted['relevance_real'].values
    y_score_modelo = df_modelo_sorted['model_score'].values
    
    # EVALUACIÓN BASELINE (Popularidad Global) 
    df_baseline_sorted = df_user.sort_values(by='product_global_popularity', ascending=False).head(10)
    y_true_baseline = df_baseline_sorted['relevance_real'].values
    y_score_baseline = df_baseline_sorted['product_global_popularity'].values
    
    # Calcular NDCG@10 usando sklearn 
    if len(y_true_modelo) > 1 and np.sum(y_true_modelo) > 0:
        ndcg_mod = ndcg_score([y_true_modelo], [y_score_modelo], k=10)
        ndcg_base = ndcg_score([y_true_baseline], [y_score_baseline], k=10)
        ndcg_modelo.append(ndcg_mod)
        ndcg_baseline.append(ndcg_base)
    else:
        ndcg_modelo.append(0.0)
        ndcg_baseline.append(0.0)
        
    # Calcular MAP@10 
    map_modelo.append(calcular_map_at_k(y_true_modelo, k=10))
    map_baseline.append(calcular_map_at_k(y_true_baseline, k=10))

# RESULTADOS
resultados = pd.DataFrame({
    'Métrica': ['NDCG@10', 'MAP@10'], 
    'Baseline (Popularidad Global)': [np.mean(ndcg_baseline), np.mean(map_baseline)],
    'XGBRanker Entrenado': [np.mean(ndcg_modelo), np.mean(map_modelo)]
})

print(resultados.to_string(index=False))


Cargando set de test filtrado por la muestra de usuarios...
Ground truth estructurado para 329 usuarios en el set de test.
Generando predicciones del XGBRanker y Baseline
Métrica  Baseline (Popularidad Global)  XGBRanker Entrenado
NDCG@10                       0.399218             0.459866
 MAP@10                       0.336483             0.332596


# Etapa 2 — Embeddings y similitud semántica

In [8]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time  # Importamos time para medir la eficiencia del código
import gc

print("Etapa 2: Embeddings y Similitud Semántica")

# Carga
ruta_carpeta = r"D:\Tipti\Data"
df_products = pd.read_csv(os.path.join(ruta_carpeta, "products.csv"))

print(f"Catálogo cargado: {len(df_products)} productos detectados.")

# CREAR LOS EMBEDDINGS 
print("Cargando el modelo de lenguaje de Sentence-Transformers")
model = SentenceTransformer('all-MiniLM-L6-v2')

#Generando vectores para los nombres de los productos
product_names = df_products['product_name'].tolist()

# Tomamos el tiempo de lo que tarda en vectorizar todo el catálogo c
inicio_emb = time.time()
product_embeddings = model.encode(product_names, show_progress_bar=True, convert_to_numpy=True)
fin_emb = time.time()

print(f"Matriz de embeddings completada con éxito en {fin_emb - inicio_emb:.2f} segundos. Forma: {product_embeddings.shape}")

# FUNCIÓN 
def get_similar_products(product_id, k=10):
    """
    Busca los K productos más similares en base al ID de un producto.
    Retorna un DataFrame con los resultados y sus scores de similitud coseno.
    """
    # Iniciar cronómetro de la consulta
    inicio_consulta = time.time()
    
    idx_query = df_products[df_products['product_id'] == product_id].index
    
    if len(idx_query) == 0:
        return f"Error: El product_id {product_id} no existe en el catálogo."
    
    idx_query = idx_query[0]
    nombre_query = df_products.loc[idx_query, 'product_name']
    
    # Extraer el vector del producto consultado
    embedding_query = product_embeddings[idx_query].reshape(1, -1)
    
    # Calcular la similitud coseno contra todo el catálogo (Vectorizado)
    scores = cosine_similarity(embedding_query, product_embeddings)[0]
    
    # índices de los productos con mayor score (excluyendo el mismo producto)
    top_indices = np.argsort(scores)[::-1][1:k+1]
    
    resultados = df_products.iloc[top_indices].copy()
    resultados['similarity_score'] = scores[top_indices]
    
    # Detener cronómetro
    fin_consulta = time.time()
    tiempo_total_ms = (fin_consulta - inicio_consulta) * 1000 # Convertimos a milisegundos
    
    print(f"\nBuscando similares para: '{nombre_query}' (ID: {product_id})")
    print(f"Tiempo de consulta: {tiempo_total_ms:.2f} ms ({(tiempo_total_ms/1000):.4f} segundos)")
    
    return resultados[['product_id', 'product_name', 'department_id', 'similarity_score']]

# FUNCIÓN 
def search_by_keyword(keyword, k=10):
    """
    Permite ingresar una palabra o frase libre (ej. 'queso crema')
    y busca los productos semánticamente más cercanos.
    """
    # Iniciar cronómetro de la consulta
    inicio_consulta = time.time()
    
    # Convertir la frase del usuario en un vector usando el mismo modelo
    embedding_query = model.encode([keyword]).reshape(1, -1)
    
    # Similitud coseno
    scores = cosine_similarity(embedding_query, product_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:k]
    
    resultados = df_products.iloc[top_indices].copy()
    resultados['similarity_score'] = scores[top_indices]
    
    # Detener cronómetro
    fin_consulta = time.time()
    tiempo_total_ms = (fin_consulta - inicio_consulta) * 1000 # Convertimos a milisegundos
    
    print(f"\nResultados semánticos para la consulta: '{keyword}'")
    print(f"Tiempo de consulta: {tiempo_total_ms:.2f} ms ({(tiempo_total_ms/1000):.4f} segundos)")
    
    return resultados[['product_id', 'product_name', 'department_id', 'similarity_score']]

c:\Users\geoco\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Etapa 2: Embeddings y Similitud Semántica
Catálogo cargado: 49688 productos detectados.
Cargando el modelo de lenguaje de Sentence-Transformers


Batches: 100%|██████████| 1553/1553 [01:26<00:00, 17.97it/s]


Matriz de embeddings completada con éxito en 86.67 segundos. Forma: (49688, 384)


## Prueba del entrenamiento

In [9]:
#Similitud por id del producto
try:
    print(get_similar_products(product_id=3, k=10))
except Exception as e:
    print(e)

print("\n" + "="*60)

# Similitus por nombre del producto
print(search_by_keyword("Gluten Free Quinoa Three Cheese", k=3))


Buscando similares para: 'Robust Golden Unsweetened Oolong Tea' (ID: 3)
Tiempo de consulta: 124.16 ms (0.1242 segundos)
       product_id                       product_name  department_id  \
41041       41042      Unsweetened Golden Oolong Tea              7   
5079         5080        Unsweetened Oolong Tea Shot              7   
45562       45563                         Oolong Tea              7   
8312         8313                 Oolong Tea Classic              7   
46245       46246                 Organic Oolong Tea              7   
35706       35707               100% Pure Oolong Tea              7   
16405       16406              Unsweetened Black Tea              7   
3720         3721              Unsweetened Green Tea              7   
36693       36694  White Tea Unsweetened Hint O'Mint              7   
20932       20933         Unsweetened Mint Green Tea              7   

       similarity_score  
41041          0.934257  
5079           0.786221  
45562          0.74

# Etapa 3 — Agente conversacional 


In [10]:

print("Consolidando funciones nativas para el Agente")

# FUNCIÓN 1
def get_user_history(user_id: int) -> str:
    """
    Retorna un string con los productos más comprados históricamente por el usuario.
    Maneja el caso Edge si el usuario es nuevo o no tiene historial en la muestra.
    """
    # Buscamos al usuario en las variables de la Etapa 1
    if 'df_dataset_final' not in globals():
        return "Error técnico: El dataset histórico no se encuentra cargado en memoria."
        
    user_data = df_dataset_final[df_dataset_final['user_id'] == user_id]
    
    # CASO EDGE: El usuario no existe en la muestra de entrenamiento
    if user_data.empty:
        return f"Caso Edge: El usuario {user_id} es nuevo en la plataforma y no cuenta con un historial de compras previo en nuestra base de datos."
    
    # Ordenamos por la frecuencia real calculada en la Etapa 1 para sugerir lo favorito
    top_compras = user_data.sort_values(by='user_product_frequency', ascending=False).head(5)
    
    # Cruzamos 
    productos_historial = pd.merge(top_compras, df_products, on='product_id', how='inner')
    
    lineas = []
    for _, row in productos_historial.iterrows():
        lineas.append(f"- ID Producto: {row['product_id']} | Nombre: {row['product_name']} (Veces comprado: {int(row['user_product_frequency'])})")
        
    return f"Historial de compras del usuario {user_id}:\n" + "\n".join(lineas)


# FUNCIÓN 2
def get_similar_products_agent(product_id: int, k: int = 3) -> str:
    """
    Llama directamente a el modelo de embeddings de la Etapa 2.
    Busca los K productos semánticamente más parecidos y los devuelve formateados en texto.
    """
    # CASO EDGE: El producto no existe en el catálogo general
    idx_query = df_products[df_products['product_id'] == product_id].index
    if len(idx_query) == 0:
        return f"Caso Edge: El product_id {product_id} no fue encontrado en el catálogo maestro."
    
    # Reutilizamos la matriz 'product_embeddings' de la Etapa 2
    idx_query = idx_query[0]
    embedding_query = product_embeddings[idx_query].reshape(1, -1)
    
    # Similitud coseno 
    scores = cosine_similarity(embedding_query, product_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][1:k+1]
    
    resultados = df_products.iloc[top_indices]
    scores_top = scores[top_indices]
    
    lineas = []
    for (_, row), score in zip(resultados.iterrows(), scores_top):
        lineas.append(f"- ID: {row['product_id']} | Nombre: {row['product_name']} | Similitud Semántica: {score:.2f}")
        
    return f"Productos similares al ID {product_id}:\n" + "\n".join(lineas)


#  FUNCIÓN 3:
def predict_reorder(user_id: int, product_id: int) -> str:
    """
    Llama al XGBRanker entrenado en la Etapa 1 para calcular la probabilidad/score
    de que el usuario vuelva a comprar (reorder) un artículo específico.
    """
    if 'ranker' not in globals() or 'features' not in globals():
        return "Error técnico: El modelo clasificador XGBRanker no está inicializado en la memoria del Kernel."
        
    # Buscamos la fila con las features calculadas para este par usuario-producto específico
    fila = df_dataset_final[(df_dataset_final['user_id'] == user_id) & (df_dataset_final['product_id'] == product_id)]
    
    # CASO EDGE: El usuario nunca ha comprado ese producto, por lo que el modelo no tiene features directas
    if fila.empty:
        return f"El usuario {user_id} nunca ha interactuado con el producto {product_id}. Score de Reordenamiento estimado por regla: 0.0000"
    
    # Extraemos el vector de características con el que se entrenó el modelo tabular
    X_inferencia = fila[features]
    
    # el score numérico
    score_xgboost = ranker.predict(X_inferencia)[0]
    
    return f"El modelo XGBRanker calculó un score de relevancia de reorden de: {score_xgboost:.4f} para el usuario {user_id} y el producto {product_id}."

print("Las 3 funciones lógicas se han creado de forma exitosa")

Consolidando funciones nativas para el Agente
Las 3 funciones lógicas se han creado de forma exitosa


## Prueba

In [12]:
print(get_user_history(user_id=99989))
print("-" * 50)
#___________________________________________

print(get_similar_products_agent(product_id=11, k=3))
print("-" * 50)

#___________________________________________

print(predict_reorder(user_id=99989, product_id=31720))
print("-" * 50)

#___________________________________________

print(get_user_history(user_id=99989))

Historial de compras del usuario 99989:
- ID Producto: 3957 | Nombre: 100% Raw Coconut Water (Veces comprado: 8)
- ID Producto: 31720 | Nombre: Organic  Whole Milk (Veces comprado: 8)
- ID Producto: 1194 | Nombre: Natural Artisan Water (Veces comprado: 6)
- ID Producto: 21709 | Nombre: Sparkling Lemon Water (Veces comprado: 6)
- ID Producto: 691 | Nombre: Organic Promise Strawberry Fields Cereal (Veces comprado: 5)
--------------------------------------------------
Productos similares al ID 11:
- ID: 18076 | Nombre: Orange Peach Mango Juice | Similitud Semántica: 0.96
- ID: 31690 | Nombre: Peach Mangosteen Juice Drink | Similitud Semántica: 0.91
- ID: 7687 | Nombre: Strawberry Peach Juice | Similitud Semántica: 0.90
--------------------------------------------------
El modelo XGBRanker calculó un score de relevancia de reorden de: 5.3909 para el usuario 99989 y el producto 31720.
--------------------------------------------------
Historial de compras del usuario 99989:
- ID Producto: 3

## Creación del bot


In [18]:
%pip install google-genai

     ---------------------------------------- 0.0/832.5 kB ? eta -:--:--
     ----- -------------------------------- 112.6/832.5 kB 3.3 MB/s eta 0:00:01
     ---------------- --------------------- 368.6/832.5 kB 4.6 MB/s eta 0:00:01
     --------------------------- ---------- 593.9/832.5 kB 4.7 MB/s eta 0:00:01
     -------------------------------------  829.4/832.5 kB 5.3 MB/s eta 0:00:01
     -------------------------------------- 832.5/832.5 kB 4.8 MB/s eta 0:00:00
     ---------------------------------------- 0.0/472.3 kB ? eta -:--:--
     ----------------------------- -------- 368.6/472.3 kB 7.6 MB/s eta 0:00:01
     -------------------------------------- 472.3/472.3 kB 7.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/178.7 kB ? eta -:--:--
     ------------------------------------- 178.7/178.7 kB 11.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/73.1 kB ? eta -:--:--
     ---------------------------------------- 73.1/73.1 kB 4.2 MB/s e


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os
import time  # Importamos time para controlar el rate limit
from google import genai
from google.genai import types

system_instruction = """
Eres un agente conversacional inteligente y experto en recomendaciones para un supermercado e-commerce (estilo Instacart o Tipti).
Tu objetivo es ayudar a los usuarios a armar carritos de compra eficientes utilizando las tres herramientas (Tools) que tienes conectadas.

Reglas obligatorias de comportamiento y encadenamiento:
1. Cuando un usuario se identifique con su ID, DEBES llamar primero a 'get_user_history' para conocer sus hábitos.
2. Basado en los IDs de su historial, busca productos similares usando 'get_similar_products_agent' para ofrecerle alternativas lógicas.
3. Para asegurar que la recomendación sea ideal, valida la afinidad o score de probabilidad llamando a 'predict_reorder'.
4. Coordina y encadena las herramientas de forma automática en una sola respuesta conversacional.
5. Responde siempre en español, con un tono extremadamente profesional, claro, estructurado con viñetas y amigable.
6. Maneja los casos Edge con transparencia: si un usuario es nuevo o un producto no existe en el catálogo, explícaselo cordialmente al usuario.
"""

# Wrappers con delay para respetar el plan gratuito de Google (5 RPM)
def get_user_history_delayed(user_id: int) -> str:
    time.sleep(4)  # Pausa de 4 segundos para no saturar la cuota gratis
    return get_user_history(user_id)

def get_similar_products_agent_delayed(product_id: int, k: int = 3) -> str:
    time.sleep(4)  
    return get_similar_products_agent(product_id, k)

def predict_reorder_delayed(user_id: int, product_id: int) -> str:
    time.sleep(4)  
    return predict_reorder(user_id, product_id)


def chatear_con_tipti_bot(prompt_usuario: str):
    print(f"Usuario: '{prompt_usuario}'")
    print("Pensando...\n")
    
    try:
        client = genai.Client()
        
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt_usuario,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                # Pasamos las funciones con delay integrado 
                tools=[get_user_history_delayed, get_similar_products_agent_delayed, predict_reorder_delayed],
                temperature=0.2
            )
        )
        
        print("Respuesta Final del Bot:")
        print("====================================================================")
        print(response.text)
        print("====================================================================")
        
    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("[Quota Limit] Se alcanzó el límite de solicitudes por minuto de Google AI Studio.")
            print("Por favor, espera 15 segundos y vuelve a ejecutar esta celda.")
        else:
            print(f"Error al conectar con la API en vivo: {e}")

print("Terminado con éxito.")

Terminado con éxito.


## Prueba

In [14]:
chatear_con_tipti_bot(
    "Hola, soy el usuario 99989. Quiero armar mi cesta de compras habitual y "
    "que me recomiendes opciones similares verificando su score con el modelo tabular."
)

print("\n" + "="*80 + "\n")

chatear_con_tipti_bot(
    "Hola, soy el usuario 99989. ¿Qué me recomiendas comprar en base a mis compras pasadas?"
)

Usuario: 'Hola, soy el usuario 99989. Quiero armar mi cesta de compras habitual y que me recomiendes opciones similares verificando su score con el modelo tabular.'
Pensando...

Error al conectar con la API en vivo: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.


Usuario: 'Hola, soy el usuario 99989. ¿Qué me recomiendas comprar en base a mis compras pasadas?'
Pensando...

Error al conectar con la API en vivo: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.
